# Lezione Pratica 4 — Gamut e Profili Colore

**Corso:** Trattamento di Dati Multimediali  
**Riferimento teorico:** Lezione 4 (Scienza dei Colori) · Lezione 5 (Modelli di Colore nelle Immagini)

**Obiettivi della lezione:**
- Visualizzare il diagramma di cromaticità CIE 1931 xy e collocare in esso i gamut di sRGB, AdobeRGB e DCI-P3
- Comprendere la *device-dependence* dello spazio RGB e la necessità dei profili colore
- Applicare la correzione gamma sRGB e verificarne empiricamente gli effetti
- Convertire immagini tra profili colore con Pillow `ImageCms`
- Rilevare e visualizzare i colori *out-of-gamut* rispetto a sRGB
- Confrontare sRGB e L*a*b* come spazi di misura della differenza colore (ΔE)


---
## 1. Setup


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import colour
from PIL import Image, ImageCms
import urllib.request, os

IMG_URL  = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"
IMG_PATH = "lena.png"

if not os.path.exists(IMG_PATH):
    try:
        urllib.request.urlretrieve(IMG_URL, IMG_PATH)
        print(f"Immagine scaricata: {IMG_PATH}")
    except Exception as e:
        print(f"Download fallito ({e}). Genero immagine sintetica.")
        img_fb = Image.new("RGB", (512, 512))
        px = img_fb.load()
        for i in range(512):
            for j in range(512):
                px[i, j] = (i // 2, j // 2, (i + j) // 4)
        img_fb.save(IMG_PATH)

img = Image.open(IMG_PATH).convert("RGB")
print(f"Immagine caricata: {img.size}, modalità: {img.mode}")


ModuleNotFoundError: No module named 'colour'

---
## 2. Il diagramma di cromaticità CIE 1931

Il diagramma di cromaticità CIE 1931 rappresenta tutti i colori percepibili dall'osservatore umano
standard in uno spazio bidimensionale $(x, y)$, ottenuto proiettando lo spazio tristimolo XYZ:

$$x = \frac{X}{X+Y+Z}, \quad y = \frac{Y}{X+Y+Z}$$

Il contorno a ferro di cavallo (**locus dello spettro**) raccoglie i colori monocromatici puri.
Tutti i colori reali percepibili cadono all'interno di quest'area.

> **Richiamo dalla teoria (Lezione 4):**  
> Lo spazio CIE XYZ è *device-independent*: descrive il colore come lo percepisce l'occhio umano,
> indipendentemente dal dispositivo. Al contrario, uno spazio RGB è sempre *device-dependent*
> perché dipende dalle cromaticità delle primarie fisiche del dispositivo.


In [ ]:
# ── Locus dello spettro visibile ─────────────────────────────────────────────
cmfs = colour.colorimetry.MSDS_CMFS['CIE 1931 2 Degree Standard Observer']
wl   = cmfs.wavelengths
XYZ  = cmfs.values
X, Y, Z = XYZ[:, 0], XYZ[:, 1], XYZ[:, 2]
total    = X + Y + Z
mask     = total > 1e-10
x_locus  = np.where(mask, X / total, 0.0)
y_locus  = np.where(mask, Y / total, 0.0)

# ── Gamut di tre spazi colore ─────────────────────────────────────────────────
spaces = {
    'sRGB':            ('tab:blue',   colour.RGB_COLOURSPACES['sRGB']),
    'Adobe RGB (1998)':('tab:orange', colour.RGB_COLOURSPACES['Adobe RGB (1998)']),
    'DCI-P3':          ('tab:green',  colour.RGB_COLOURSPACES['DCI-P3']),
}

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(x_locus, y_locus, 'k-', linewidth=1.2)
ax.plot([x_locus[-1], x_locus[0]], [y_locus[-1], y_locus[0]], 'k--',
        linewidth=0.8, label='Linea dei viola')

for target_wl in [460, 480, 500, 520, 550, 580, 620, 700]:
    idx = np.argmin(np.abs(wl - target_wl))
    ax.annotate(f'{target_wl}', (x_locus[idx], y_locus[idx]),
                fontsize=7, color='gray',
                xytext=(x_locus[idx] * 1.04, y_locus[idx] * 1.02))

for name, (color, cs) in spaces.items():
    prim = cs.primaries
    wp   = cs.whitepoint
    tri  = plt.Polygon(prim, closed=True, fill=True,
                        facecolor=color, alpha=0.15,
                        edgecolor=color, linewidth=2, label=name)
    ax.add_patch(tri)
    for p in prim:
        ax.plot(*p, 'o', color=color, markersize=6)
    ax.plot(*wp, '+', color=color, markersize=10, markeredgewidth=2)

ax.set_xlim(-0.05, 0.80); ax.set_ylim(-0.05, 0.92)
ax.set_xlabel('x', fontsize=12); ax.set_ylabel('y', fontsize=12)
ax.set_title('Diagramma di cromaticità CIE 1931\nGamut di sRGB, Adobe RGB e DCI-P3', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.set_aspect('equal')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


**Cosa osservare:**
- Il gamut di **sRGB** (blu) copre circa il 35% del diagramma CIE: è lo standard per il web e i monitor consumer.
- **Adobe RGB** (arancione) ha la primaria verde spostata verso $y$ maggiore, coprendo verde e ciano — usato in fotografia professionale.
- **DCI-P3** (verde) è lo standard cinema; la primaria rossa è a lunghezze d'onda più alte.
- Il **punto bianco** (croce) di ciascuno spazio cade vicino a D65 ($x≈0.313$, $y≈0.329$).
- I colori sul bordo del locus (monocromatici puri) non sono riproducibili da nessuno spazio RGB.


---
## 3. Gamma Correction

I display hanno una risposta non lineare tra segnale digitale e luminanza emessa (legge di potenza
con $\gamma \approx 2.2$). Per compensarla, i valori sRGB vengono *pre-codificati* con la funzione inversa
(*encoding gamma*). La funzione sRGB è a tratti: lineare per valori bassi, poi con esponente $1/2.4$.

> **Richiamo dalla teoria (Lezione 4, Sezione 5):**  
> La correzione gamma serve a pre-compensare la non-linearità dei display. Senza di essa,
> le immagini risulterebbero troppo scure, con dettaglio perso nelle ombre.


In [ ]:
def srgb_encode(linear):
    """Luminanza lineare [0,1] → valore sRGB codificato [0,1]."""
    return np.where(linear <= 0.0031308,
                    12.92 * linear,
                    1.055 * linear ** (1 / 2.4) - 0.055)

def srgb_decode(encoded):
    """Valore sRGB codificato [0,1] → luminanza lineare [0,1]."""
    return np.where(encoded <= 0.04045,
                    encoded / 12.92,
                    ((encoded + 0.055) / 1.055) ** 2.4)

x_lin = np.linspace(0, 1, 512)
x_enc = srgb_encode(x_lin)
x_pow = x_lin ** (1 / 2.2)   # gamma pura 2.2 per confronto

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(x_lin, x_lin,   'k--', linewidth=1,   label='Identità (nessuna correzione)')
axes[0].plot(x_lin, x_pow,   color='tab:orange', linewidth=2, label='Gamma pura 1/2.2')
axes[0].plot(x_lin, x_enc,   color='tab:blue',   linewidth=2, label='sRGB encoding')
axes[0].set_xlabel('Luminanza lineare (input)', fontsize=11)
axes[0].set_ylabel('Valore codificato (output)', fontsize=11)
axes[0].set_title('Funzione di encoding gamma sRGB', fontsize=12)
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

arr_orig = np.array(img, dtype=np.float32) / 255.0
arr_enc  = (srgb_encode(arr_orig) * 255).clip(0, 255).astype(np.uint8)
arr_dec  = (srgb_decode(arr_orig) * 255).clip(0, 255).astype(np.uint8)

axes[1].hist(arr_orig.flatten() * 255, bins=64, color='gray',       alpha=0.6,
             label='Originale', density=True)
axes[1].hist(arr_enc.flatten(),         bins=64, color='tab:blue',   alpha=0.6,
             label='Dopo encoding (schiarita)', density=True)
axes[1].hist(arr_dec.flatten(),         bins=64, color='tab:orange', alpha=0.6,
             label='Dopo decoding / linearizzazione', density=True)
axes[1].set_xlabel('Valore pixel', fontsize=11)
axes[1].set_ylabel('Densità', fontsize=11)
axes[1].set_title('Distribuzione pixel: effetto gamma correction', fontsize=12)
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
img_enc = Image.fromarray(arr_enc)
img_dec = Image.fromarray(arr_dec)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, im, title in zip(axes,
    [img, img_enc, img_dec],
    ['Originale', 'Encoding gamma (schiarita)', 'Decoding / linearizzazione (scurita)']):
    ax.imshow(im); ax.set_title(title, fontsize=11); ax.axis('off')

plt.suptitle('Effetto visivo della gamma correction', fontsize=13)
plt.tight_layout()
plt.show()

roundtrip_err = np.abs(srgb_decode(srgb_encode(arr_orig)) - arr_orig).max()
print(f"Errore massimo roundtrip encode→decode: {roundtrip_err:.2e}  (atteso: ~0)")


---
## 4. Profili colore con Pillow `ImageCms`

Un **profilo ICC** descrive le caratteristiche colorimetriche di un dispositivo e permette conversioni
accurate tra spazi colore via uno spazio di riferimento intermedio (CIELAB o CIEXYZ).

Pillow espone queste operazioni con `ImageCms`:
- `createProfile(name)` — profilo predefinito (`'sRGB'`, `'LAB'`)
- `buildTransform(src, dst, src_mode, dst_mode)` — prepara la trasformazione
- `applyTransform(image, transform)` — applica la trasformazione

> **Richiamo dalla teoria (Lezione 5, Sezione 4):**  
> sRGB è definito con una matrice di conversione verso CIE XYZ e una funzione gamma standard.
> Funziona da *spazio di scambio* condiviso tra dispositivi diversi.


In [ ]:
srgb_profile = ImageCms.createProfile('sRGB')
lab_profile  = ImageCms.createProfile('LAB', colorTemp=6500)

tf_to_lab = ImageCms.buildTransform(srgb_profile, lab_profile, 'RGB', 'LAB')
img_lab   = ImageCms.applyTransform(img, tf_to_lab)

print(f"Originale:   modalità={img.mode},     size={img.size}")
print(f"In L*a*b*:   modalità={img_lab.mode}, size={img_lab.size}")

L, a, b  = img_lab.split()
L_sc = np.array(L) / 255.0 * 100                   # [0, 100]
a_sc = np.array(a).astype(np.int16) - 128           # [-128, 127]
b_sc = np.array(b).astype(np.int16) - 128

print(f"\nL*  media={L_sc.mean():.1f},  min={L_sc.min():.1f},  max={L_sc.max():.1f}")
print(f"a*  media={a_sc.mean():.1f},  min={a_sc.min()},   max={a_sc.max()}")
print(f"b*  media={b_sc.mean():.1f},  min={b_sc.min()},   max={b_sc.max()}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

arrs    = [L_sc,   a_sc,    b_sc]
names   = ['L* (luminanza percettiva)', 'a* (verde ↔ rosso)', 'b* (blu ↔ giallo)']
cmaps   = ['gray', 'RdYlGn', 'coolwarm']
vranges = [(0,100), (-128,127), (-128,127)]

for col, (arr, name, cmap, vr) in enumerate(zip(arrs, names, cmaps, vranges)):
    im = axes[0, col].imshow(arr, cmap=cmap, vmin=vr[0], vmax=vr[1])
    axes[0, col].set_title(name, fontsize=11); axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], fraction=0.046, pad=0.04)

    axes[1, col].hist(arr.flatten(), bins=60, color='steelblue',
                      edgecolor='white', alpha=0.85)
    axes[1, col].axvline(arr.mean(), color='red', linestyle='--', linewidth=1.5,
                          label=f'Media = {arr.mean():.1f}')
    axes[1, col].set_xlabel('Valore', fontsize=10)
    axes[1, col].set_ylabel('Frequenza', fontsize=10)
    axes[1, col].set_title(f'Istogramma {name.split()[0]}', fontsize=11)
    axes[1, col].legend(fontsize=9); axes[1, col].grid(True, alpha=0.3)

plt.suptitle("Analisi dell'immagine nello spazio L*a*b*", fontsize=13)
plt.tight_layout()
plt.show()


---
## 5. Colori Out-of-Gamut

Un colore è **out-of-gamut** quando non può essere riprodotto fedelmente da un dispositivo.
Accade spesso quando si converte da uno spazio più ampio (AdobeRGB) a uno più ristretto (sRGB),
o quando si stampa un'immagine pensata per monitor.

> **Richiamo dalla teoria (Lezione 5, Sezione 7):**  
> Il gamut di una stampante CMYK è generalmente più piccolo di quello di un display RGB.
> I colori out-of-gamut vengono *compressi* verso il confine del gamut riproducibile (gamut mapping).


In [ ]:
# ── Posizione di colori campione sul diagramma CIE xy ─────────────────────────
test_rgb = np.array([
    [255,   0,   0], [  0, 255,   0], [  0,   0, 255],
    [255, 255,   0], [  0, 255, 255], [255,   0, 255],
    [255, 128,   0], [  0, 200, 100], [ 80,   0, 255],
    [255,  20,  60], [255, 255, 255], [  0,   0,   0],
], dtype=np.float32) / 255.0

color_names = ['Rosso','Verde','Blu','Giallo','Ciano','Magenta',
               'Arancione','Verde-turchese','Viola','Rosso fragola',
               'Bianco','Nero']

XYZ_t = colour.sRGB_to_XYZ(test_rgb)
tot_t = XYZ_t.sum(axis=1, keepdims=True)
tot_t = np.where(tot_t < 1e-10, 1.0, tot_t)
xy_t  = XYZ_t[:, :2] / tot_t

fig, ax = plt.subplots(figsize=(9, 8))
ax.plot(x_locus, y_locus, 'k-', linewidth=1.2)
ax.plot([x_locus[-1], x_locus[0]], [y_locus[-1], y_locus[0]], 'k--', linewidth=0.8)

srgb_cs = colour.RGB_COLOURSPACES['sRGB']
ax.add_patch(plt.Polygon(srgb_cs.primaries, closed=True,
    fill=True, facecolor='tab:blue', alpha=0.10,
    edgecolor='tab:blue', linewidth=2, label='Gamut sRGB'))

for xy, name, rgb in zip(xy_t, color_names, test_rgb):
    mc  = tuple(rgb.tolist())
    ec  = 'black' if rgb.mean() > 0.85 else mc
    ax.scatter(*xy, color=mc, edgecolors=ec, s=120, linewidths=1.5, zorder=5)
    ax.annotate(name, xy, fontsize=7.5, ha='left',
                xytext=(xy[0]+0.01, xy[1]+0.01))

ax.set_xlim(-0.05, 0.80); ax.set_ylim(-0.05, 0.92)
ax.set_xlabel('x', fontsize=12); ax.set_ylabel('y', fontsize=12)
ax.set_title('Posizione dei colori campione nel diagramma CIE xy\n'
             'rispetto al gamut sRGB', fontsize=13)
ax.legend(fontsize=10); ax.set_aspect('equal')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# ── Mappa out-of-gamut sull'immagine ─────────────────────────────────────────
arr_lin = srgb_decode(np.array(img, dtype=np.float32) / 255.0)

M_to_XYZ   = colour.RGB_COLOURSPACES['sRGB'].matrix_RGB_to_XYZ
M_from_XYZ = colour.RGB_COLOURSPACES['sRGB'].matrix_XYZ_to_RGB

XYZ_img  = np.einsum('ij,...j->...i', M_to_XYZ, arr_lin)
rgb_back = np.einsum('ij,...j->...i', M_from_XYZ, XYZ_img)

oog_mask = np.any((rgb_back < -0.01) | (rgb_back > 1.01), axis=2)
oog_pct  = oog_mask.mean() * 100
print(f"Pixel out-of-gamut sRGB: {oog_mask.sum()} ({oog_pct:.2f}%)")

img_oog = np.array(img).copy()
img_oog[oog_mask] = [255, 0, 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(img);        axes[0].set_title('Originale', fontsize=12); axes[0].axis('off')
axes[1].imshow(img_oog);    axes[1].set_title(
    f'Out-of-gamut evidenziati in rosso ({oog_pct:.2f}% dei pixel)',
    fontsize=12); axes[1].axis('off')
plt.suptitle('Rilevazione pixel out-of-gamut sRGB', fontsize=13)
plt.tight_layout()
plt.show()


---
## 6. Uniformità percettiva: sRGB vs L\*a\*b\*

Lo spazio sRGB **non è percettivamente uniforme**: uguale differenza numerica in zone diverse
dello spazio non corrisponde alla stessa differenza percepita dall'occhio.

**CIELAB** è stato progettato per essere approssimativamente uniforme: la distanza euclidea
$\Delta E^*_{ab}$ corrisponde (approssimativamente) alla differenza percepita.


In [ ]:
N, H = 256, 60

# Gradiente lineare nei valori digitali sRGB
grad_srgb = np.tile(np.linspace(0, 255, N).astype(np.uint8), (H, 1))

# Gradiente lineare in L* [0-100] → convertito in sRGB display
L_vals = np.linspace(0, 100, N)
Y_vals = np.where(L_vals > 8, ((L_vals + 16) / 116) ** 3, L_vals / 903.3)
srgb_from_L = (srgb_encode(Y_vals) * 255).clip(0, 255).astype(np.uint8)
grad_lab    = np.tile(srgb_from_L, (H, 1))

fig, axes = plt.subplots(2, 1, figsize=(12, 4))
axes[0].imshow(np.stack([grad_srgb]*3, axis=2), aspect='auto')
axes[0].set_title('Gradiente lineare in sRGB (valori digitali uniformi)', fontsize=11)
axes[0].axis('off')
axes[1].imshow(np.stack([grad_lab]*3, axis=2), aspect='auto')
axes[1].set_title('Gradiente lineare in L* (percezione uniforme)', fontsize=11)
axes[1].axis('off')
plt.suptitle('Uniformità percettiva: sRGB vs L*', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ── Confronto ΔE tra coppie di colori ────────────────────────────────────────
def delta_E_ab(rgb1, rgb2):
    lab1 = colour.XYZ_to_Lab(colour.sRGB_to_XYZ(np.array(rgb1)))
    lab2 = colour.XYZ_to_Lab(colour.sRGB_to_XYZ(np.array(rgb2)))
    return float(np.sqrt(np.sum((lab1 - lab2) ** 2)))

pairs = [
    ([0.00, 0.00, 0.00], [0.10, 0.10, 0.10], "Nero → grigio scuro (ombre)"),
    ([0.90, 0.90, 0.90], [1.00, 1.00, 1.00], "Grigio chiaro → bianco (luci)"),
    ([0.50, 0.50, 0.50], [0.60, 0.60, 0.60], "Grigio medio +10%"),
    ([1.00, 0.00, 0.00], [0.00, 1.00, 0.00], "Rosso puro → Verde puro"),
]

print(f"{'Coppia':<45} {'ΔE_sRGB (×255)':>14} {'ΔE*ab':>8}")
print("─" * 70)
for rgb1, rgb2, desc in pairs:
    d_srgb = np.sqrt(sum((a-b)**2 for a, b in zip(rgb1, rgb2))) * 255
    d_lab  = delta_E_ab(rgb1, rgb2)
    print(f"{desc:<45} {d_srgb:>14.1f} {d_lab:>8.1f}")

print("\nNota: ΔE*ab < 1 → impercettibile; > 3 → visibilmente diverso.")


---
## 7. Riepilogo e Concetti Chiave

| Concetto | Cosa abbiamo visto |
|---|---|
| **Diagramma CIE xy** | Rappresenta tutti i colori percepibili; i gamut RGB sono triangoli al suo interno |
| **Device-dependence** | Lo stesso valore RGB produce colori diversi su dispositivi con primarie diverse |
| **Gamut** | sRGB ⊂ AdobeRGB ⊂ DCI-P3 ⊂ (colori percepibili); non tutti i colori visibili sono riproducibili |
| **Gamma correction** | Encoding schiarisce per compensare la non-linearità del display; decoding linearizza |
| **Profilo ICC** | Descrive le caratteristiche colorimetriche di un dispositivo; `ImageCms` consente conversioni accurate |
| **L\*a\*b\*** | Spazio percettivamente uniforme; L\* misura la luminosità, a\* e b\* la crominanza |
| **Out-of-gamut** | Colori non riproducibili; vengono compressi al confine del gamut (gamut mapping) |
| **ΔE\*ab** | Metrica percettiva; uguale ΔE numerico in sRGB ≠ uguale differenza percepita |

---

### Domande di riflessione

1. Perché un'immagine sRGB visualizzata su un monitor AdobeRGB senza gestione del colore appare sovrasatura?
2. Cosa succede ai colori out-of-gamut quando si converte un'immagine AdobeRGB in sRGB per il web?
3. Perché L\*a\*b\* è preferito a sRGB per confrontare e misurare differenze di colore?
4. In che modo la gamma correction influisce sulla qualità visiva nelle zone di ombra?
